# Notebook 04 — Modélisation
## Phase 3 : Modélisation, rééquilibrage et comparaison des modèles

Objectif : entraîner au moins 4 familles de modèles, tester au moins 3 stratégies de gestion du déséquilibre, et identifier la meilleure combinaison modèle × stratégie selon la métrique F1.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
import joblib

PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data' / 'processed'
MODEL_DIR = PROJECT_ROOT / 'models'

train_df = pd.read_csv(DATA_DIR / 'train.csv')
X_train = train_df.drop(columns=['bad_nutrition'])
y_train = train_df['bad_nutrition']

preprocessor = joblib.load(MODEL_DIR / 'preprocessor.joblib')

print('Train shape:', X_train.shape)
print('Target distribution:')
print(y_train.value_counts(normalize=True).mul(100).round(2))

Train shape: (9068, 16)
Target distribution:
bad_nutrition
0    82.49
1    17.51
Name: proportion, dtype: float64


## 1. Modèles sélectionnés

Nous testons quatre familles de modèles diversifiées :
- Régression logistique (baseline linéaire, class_weight)
- Arbre de décision (non-linéaire, interprétable)
- Forêt aléatoire (ensemble, robuste)
- Perceptron multicouche (MLP, réseau de neurones tabulaire)

Ces quatre types couvrent des approches linéaires, arborescentes, ensemblistes et neuronales.

In [3]:
models = {
    'LogisticRegression': LogisticRegression(random_state=42, solver='liblinear', max_iter=5000, class_weight='balanced'),
    'DecisionTree': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    'RandomForest': RandomForestClassifier(random_state=42, n_estimators=200, class_weight='balanced_subsample'),
    'MLP': MLPClassifier(random_state=42, max_iter=500, early_stopping=True)
}

strategies = {
    'baseline': None,
    'smote': SMOTE(random_state=42),
    'undersample': RandomUnderSampler(random_state=42)
}

def build_pipeline(model, sampler=None):
    preproc = clone(preprocessor)
    if sampler is None:
        return Pipeline([('preprocessor', preproc), ('clf', clone(model))])
    return ImbPipeline([('preprocessor', preproc), ('sampler', sampler), ('clf', clone(model))])

def evaluate_pipeline(pipeline):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1', n_jobs=1)
    return scores.mean(), scores.std()

In [5]:
results = []
for model_name, model in models.items():
    for strategy_name, sampler in strategies.items():
        pipe = build_pipeline(model, sampler)
        mean_f1, std_f1 = evaluate_pipeline(pipe)
        results.append({
            'model': model_name,
            'strategy': strategy_name,
            'mean_f1': mean_f1,
            'std_f1': std_f1
        })
        print(f'{model_name:<15} | {strategy_name:<10} | F1 = {mean_f1:.4f} ± {std_f1:.4f}')

results_df = pd.DataFrame(results).sort_values(by=['mean_f1', 'std_f1'], ascending=[False, True]).reset_index(drop=True)
results_df.to_csv(MODEL_DIR / 'model_selection_results.csv', index=False)

print('Meilleure configuration :')
print(results_df.head(1).to_string(index=False))

LogisticRegression | baseline   | F1 = 0.6952 ± 0.0191
LogisticRegression | smote      | F1 = 0.7008 ± 0.0209
LogisticRegression | undersample | F1 = 0.6806 ± 0.0156
DecisionTree    | baseline   | F1 = 0.8139 ± 0.0329
DecisionTree    | smote      | F1 = 0.8143 ± 0.0206
DecisionTree    | undersample | F1 = 0.7757 ± 0.0247
RandomForest    | baseline   | F1 = 0.8542 ± 0.0256
RandomForest    | smote      | F1 = 0.8638 ± 0.0203
RandomForest    | undersample | F1 = 0.8186 ± 0.0198
MLP             | baseline   | F1 = 0.7635 ± 0.0169
MLP             | smote      | F1 = 0.8059 ± 0.0103
MLP             | undersample | F1 = 0.7120 ± 0.0208
Meilleure configuration :
       model strategy  mean_f1   std_f1
RandomForest    smote 0.863845 0.020317


## 2. Analyse et choix du meilleur modèle

Les résultats F1±σ pour les 12 configurations sont sauvegardés dans `models/model_selection_results.csv`.
Le modèle retenu pour l’optimisation en Phase 3 sera celui qui maximise la moyenne F1.

